# Selezione Dinamica della Dimensione del Campione — Riconoscimento della nota (NSynth)

Notebook di riproduzione del **secondo esperimento** della **Sezione 6.5**
della tesi *"Selezione Dinamica della Dimensione del Campione in Metodi di
Ottimizzazione per il Machine Learning"*: il **riconoscimento della nota**
(pitch class, 12 classi di altezza) sul dataset NSynth.

Il notebook:

1. scarica il dataset **NSynth** (split validation e test),
2. estrae le **features chroma** (12 componenti cromatiche, media+deviazione
   standard → 24 dimensioni),
3. esegue i **quattro metodi della tesi** — codice **esattamente come nei
   listati B.1–B.4 dell'Appendice B** — su una regressione logistica
   multinomiale a 12 classi,
4. confronta i risultati con il **solutore di riferimento** scikit-learn
   (L-BFGS),
5. rigenera **figure e tabelle** usate nella tesi.

> **Riproducibilità**: seme fissato (`seed = 42`), risultati deterministici e
> coincidenti con quelli della tesi.

In [ ]:
#@title 0. Dipendenze e import
%pip install -q librosa

import os, json, time, tarfile, urllib.request
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import librosa

print("numpy", np.__version__)
print("librosa", librosa.__version__)

In [ ]:
#@title 1. Scarica ed estrae NSynth (split validation e test)
# Il train completo occupa ~24 GB: usiamo validation (12 678 clip) per
# l'addestramento e test (4 096 clip) per la valutazione. Gli strumenti dei
# due split sono disgiunti (per costruzione del dataset).

BASE = "http://download.magenta.tensorflow.org/datasets/nsynth/"

def scarica_estrai(split):
    if os.path.isdir(f"nsynth-{split}") and os.path.isdir(f"nsynth-{split}/audio"):
        print(f"OK: split {split} gia' presente")
        return
    fname = f"nsynth-{split}.jsonwav.tar.gz"
    if not os.path.exists(fname):
        print(f"Scaricamento {fname} (~{'1 GB' if split=='valid' else '350 MB'}) ...")
        urllib.request.urlretrieve(BASE + fname, fname)
    print(f"Estrazione {fname} ...")
    with tarfile.open(fname, "r:gz") as tar:
        tar.extractall()
    print(f"OK: split {split} pronto")

for s in ("valid", "test"):
    scarica_estrai(s)

import glob
print("clip valid =", len(glob.glob("nsynth-valid/audio/*.wav")))
print("clip test  =", len(glob.glob("nsynth-test/audio/*.wav")))

In [ ]:
#@title 2. Estrazione features chroma (pitch class)
SR = 16000

def estrai_chroma(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    C = librosa.feature.chroma_stft(y=y, sr=SR)          # 12 bin cromatici
    return np.concatenate([C.mean(axis=1), C.std(axis=1)])  # 24 dim

def costruisci_dati(split):
    jsondir = f"nsynth-{split}"
    with open(os.path.join(jsondir, "examples.json"), encoding="utf-8") as f:
        notes = json.load(f)
    X, y = [], []
    for note_str, md in notes.items():
        wav = os.path.join(jsondir, "audio", note_str + ".wav")
        if os.path.exists(wav):
            X.append(estrai_chroma(wav))
            y.append(md["pitch"] % 12)                   # pitch class (0=C ... 11=B)
    return np.asarray(X, np.float32), np.asarray(y, np.int64)

Xtr, Ytr = costruisci_dati("valid")
Xte, Yte = costruisci_dati("test")
NOTE = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
print("Xtr", Xtr.shape, " Xte", Xte.shape, " classi:", len(NOTE))

# standardizzazione sui dati di addestramento
mu, sd = Xtr.mean(0), Xtr.std(0); sd[sd < 1e-8] = 1.0
Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd

## Il problema di apprendimento

Il compito è la **classificazione multinomiale** della nota suonata (12
classi di altezza: C, C#, D, …, B, indipendenti dall'ottava) con una
**regressione logistica (softmax)**. Il vettore dei parametri è
$w = \operatorname{vec}(W)$, con $W \in \mathbb{R}^{12\times 25}$
(24 features + bias).

I listati B.1–B.4 della tesi operano su un generico vettore $w$ attraverso
le funzioni globali `loss_i(w, i)`, `grad_i(w, i)`, `hessvec_i(w, i, v)` e
`grad_full(w)`. Definiamo qui queste funzioni per il problema logistico
multinomiale: **gli algoritmi (celle successive) sono copiati letteralmente
dai listati e non vengono modificati**.

La regolarizzazione $L_2$ ($\lambda = 10^{-4}$) è inclusa nelle funzioni
`loss_i`/`grad_i`/`hessvec_i` per Dynamic GD, Newton-CG e BB-CCV; per
Newton-CG $L_1$ si imposta $\lambda = 0$ (la penalità $L_1$ è gestita
dall'algoritmo stesso con $\nu = 10^{-3}$).

In [ ]:
#@title 3. Definizione del problema (interfaccia usata dai listati B.1-B.4)
C = 12                      # classi (12 note)
LAM = 1e-4                  # regolarizzazione L2 (si imposta 0 per il metodo L1)
R  = 0.1                    # rapporto |H_k| / |S_k| (globale usato in B.2 e B.3)

Xtr_aug = np.hstack([Xtr, np.ones((Xtr.shape[0], 1))])   # bias come ultima colonna

# Cache su w: i listati chiamano loss_i/grad_i/hessvec_i in loop per-esempio
# sullo stesso w; le quantita' che dipendono solo da w si ricalcolano una volta.
_wc = {"w": None, "W": None, "Wnb": None}

def _prep(w):
    if _wc["w"] is not w:
        W = w.reshape(C, -1)
        _wc["w"] = w
        _wc["W"] = W
        _wc["Wnb"] = W[:, :-1]
    return _wc["W"], _wc["Wnb"]

def loss_i(w, i):
    W, Wnb = _prep(w)
    x = Xtr_aug[i]
    z = W @ x; z -= z.max()
    e = np.exp(z); p = e / e.sum()
    reg = 0.5 * LAM * np.sum(Wnb ** 2)
    return float(-np.log(max(p[Ytr[i]], 1e-15))) + reg

def grad_i(w, i):
    W, Wnb = _prep(w)
    x = Xtr_aug[i]
    z = W @ x; z -= z.max()
    e = np.exp(z); p = e / e.sum()
    ey = np.zeros(C); ey[Ytr[i]] = 1.0
    G = np.outer(p - ey, x)
    G[:, :-1] += LAM * Wnb
    return G.ravel()

def hessvec_i(w, i, v):
    W, Wnb = _prep(w)
    x = Xtr_aug[i]
    z = W @ x; z -= z.max()
    e = np.exp(z); p = e / e.sum()
    V = v.reshape(C, -1)
    q = V @ x
    r = p * q - p * (p * q).sum()
    Hv = np.outer(r, x)
    Hv[:, :-1] += LAM * V[:, :-1]
    return Hv.ravel()

def grad_full(w):                       # vettorizzato su tutto il train
    W = w.reshape(C, -1)
    Z = Xtr_aug @ W.T
    Z -= Z.max(axis=1, keepdims=True)
    P = np.exp(Z); P /= P.sum(axis=1, keepdims=True)
    Y = np.zeros_like(P); Y[np.arange(len(Ytr)), Ytr] = 1.0
    G = (P - Y).T @ Xtr_aug / len(Ytr)
    G[:, :-1] += LAM * W[:, :-1]
    return G.ravel()

N = Xtr.shape[0]                        # numero di esempi (usato dai listati)
print("N =", N, "  parametri =", C * Xtr_aug.shape[1])

## I quattro algoritmi della tesi (Appendice B)

Le celle seguenti contengono **integralmente il codice dei listati B.1–B.4
dell'Appendice B della tesi** (Dynamic GD, Newton-CG, Newton-CG $L_1$,
BB-CCV), copiato senza modifiche. Gli algoritmi operano sulle funzioni
globali `N`, `loss_i`, `grad_i`, `hessvec_i`, `grad_full` definite nella
cella precedente: **non è necessario modificare nulla** per passare dal
problema quadratico sintetico della tesi al problema reale NSynth.

#### Listato B.1 — Dynamic GD (CCV + line search di Wolfe)

In [ ]:
import numpy as np

def dynamic_gd(w0, theta, max_iter, alpha, batch0):
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history     = [w.copy().tolist()]
    batch_sizes = [n]
    for k in range(max_iter):
        indices = np.random.choice(N, size=n, replace=False)
        grads = np.array([grad_i(w, i) for i in indices])
        g = np.mean(grads, axis=0)
        if n > 1:
            var_vec = np.var(grads, axis=0, ddof=1)
        else:
            var_vec = np.zeros_like(g)
        V_norm1 = np.sum(var_vec)

        gg = np.dot(g, g)
        if gg > 1e-16:
            if V_norm1 / n > theta**2 * gg:
                n_new = int(np.ceil(V_norm1 / (theta**2 * gg))) + 1
                n = min(n_new, N)
        def J_batch(w_curr):
            return np.mean([loss_i(w_curr, i) for i in indices])

        c1 = 1e-4
        step = alpha
        J_curr = J_batch(w)
        g_norm2 = np.dot(g, g)
        def J_batch(w_curr):
            return np.mean([loss_i(w_curr, i) for i in indices])

        # Line search Wolfe
        c1, c2 = 1e-4, 0.9
        step = alpha
        J_curr = J_batch(w)
        g_norm2 = np.dot(g, g)
        d = -g
        gd = -g_norm2
        if g_norm2 > 1e-16:
            for _ in range(30):
                w_new = w + step * d
                if J_batch(w_new) <= J_curr + c1 * step * gd:
                    g_new = np.mean([grad_i(w_new, i) for i in indices], axis=0)
                    if np.dot(g_new, d) >= c2 * gd:
                        break
                step *= 0.5
            else:
                step = 0.0

        w = w + step * d

        if np.linalg.norm(grad_full(w)) < 1e-6:
            history.append(w.copy().tolist())
            batch_sizes.append(n)
            break

        history.append(w.copy().tolist())
        batch_sizes.append(n)

    return history, batch_sizes

#### Listato B.2 — Newton-CG (Hessiana sottocampionata + CG adattivo)

In [ ]:
import numpy as np

def cg(A, b, gamma, maxcg):
    x = np.zeros_like(b)
    r = b - A(x)
    p = r.copy()
    rr = np.dot(r, r)
    for _ in range(maxcg):
        Ap = A(p)
        pHp = np.dot(p, Ap)
        if pHp <= 1e-14:
            break
        alpha = rr / pHp
        x = x + alpha * p
        r_new = r - alpha * Ap
        rr_new = np.dot(r_new, r_new)
        if rr_new <= gamma * np.dot(x, x) + 1e-16:
            return x
        beta = rr_new / rr
        p = r_new + beta * p
        r = r_new
        rr = rr_new
    return x
def newton_cg(w0, theta, max_iter, alpha, batch0, R, maxcg):
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history, batch_sizes = [w.copy().tolist()], [n]
    for k in range(max_iter):
        indices_S = np.random.choice(N, size=n, replace=False)
        g = np.mean([grad_i(w, i) for i in indices_S], axis=0)
        n_h = min(max(1, int(round(R * n))), N)
        indices_H = np.random.choice(indices_S, size=n_h, replace=False)
        Hv = lambda v: np.mean([hessvec_i(w, i, v) for i in indices_H], axis=0)
        p0 = -g
        p0_norm2 = np.dot(p0, p0)
        gamma = 0.0
        if p0_norm2 > 1e-16 and n_h > 1:
            Hp0 = np.array([hessvec_i(w, i, p0) for i in indices_H])
            gamma = np.sum(np.var(Hp0, axis=0, ddof=1)) / (n_h * p0_norm2)
        d = cg(Hv, -g, gamma, maxcg)
        J_batch = lambda wc: np.mean([loss_i(wc, i) for i in indices_S])

        c1, c2 = 1e-4, 0.9
        step, J_w = alpha, J_batch(w)
        gd = np.dot(g, d)
        if gd >= 0:
            d = -g
            gd = -np.dot(g, g)
        for _ in range(30):
            w_new = w + step * d
            if J_batch(w_new) <= J_w + c1 * step * gd:
                g_new = np.mean([grad_i(w_new, i) for i in indices_S], axis=0)
                if np.dot(g_new, d) >= c2 * gd:
                    break
            step *= 0.5
        else:
            step = 0.0
        w = w + step * d
        history.append(w.copy().tolist())
        batch_sizes.append(n)
        indices_new = np.random.choice(N, size=n, replace=False)
        g_new = np.mean([grad_i(w, i) for i in indices_new], axis=0)
        var_vec = np.var([grad_i(w, i) for i in indices_new], axis=0, ddof=1) if n > 1 else np.zeros_like(g_new)
        V_norm1, gg_new = np.sum(var_vec), np.dot(g_new, g_new)
        if gg_new > 1e-16 and V_norm1 / n > theta**2 * gg_new:
            n = min(int(np.ceil(V_norm1 / (theta**2 * gg_new))) + 1, N)
        if np.linalg.norm(grad_full(w)) < 1e-6:
            break
    return history, batch_sizes

#### Listato B.3 — Newton-CG per problemi $L_1$-regolarizzati

In [ ]:
import numpy as np

def newton_l1(w0, theta, max_iter, alpha, batch0, nu, sigma, maxcg, eta=0.5):
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history     = [w.copy().tolist()]
    batch_sizes = [n]
    def F_batch(v, indices):
        Jb = np.mean([loss_i(v, i) for i in indices])
        return Jb + nu * np.sum(np.abs(v))
    def subgrad_batch(v, indices):
        grads = np.array([grad_i(v, i) for i in indices])
        gJ = np.mean(grads, axis=0)
        g = np.zeros_like(v)
        for i in range(len(v)):
            if v[i] > 0:
                g[i] = gJ[i] + nu
            elif v[i] < 0:
                g[i] = gJ[i] - nu
            else:
                if gJ[i] < -nu:
                    g[i] = gJ[i] + nu
                elif gJ[i] > nu:
                    g[i] = gJ[i] - nu
                else:
                    g[i] = 0.0
        return g
    def project_orthant(v, z):
        res = v.copy()
        for i in range(len(v)):
            if z[i] != 0 and np.sign(res[i]) != z[i]:
                res[i] = 0.0
        return res
    for k in range(max_iter):
        indices_S = np.random.choice(N, size=n, replace=False)
        grads = np.array([grad_i(w, i) for i in indices_S])
        g_batch = np.mean(grads, axis=0)
        z = np.where(w > 0, 1,
            np.where(w < 0, -1,
                np.where(g_batch < -nu, 1,
                    np.where(g_batch > nu, -1, 0))))
        sg = subgrad_batch(w, indices_S)
        sgn = np.linalg.norm(sg)
        if sgn < 1e-10:
            history.append(w.copy().tolist())
            batch_sizes.append(n)
            break
        n_h = max(1, int(round(R * n)))
        n_h = min(n_h, N)
        indices_H = np.random.choice(indices_S, size=n_h, replace=False)
        free = (z != 0)
        d = np.zeros_like(w)
        if np.any(free):
            g_free = sg[free]
            # Hessian free: prodotto H·v senza costruire H
            def Hv(v_full):
                return np.mean([hessvec_i(w, i, v_full) for i in indices_H], axis=0)
            def Hv_free(v_free):
                v_full = np.zeros_like(w)
                v_full[free] = v_free
                Hv_full = Hv(v_full)
                return Hv_full[free]
            tol_cg = eta * np.linalg.norm(g_free)
            d_free = np.zeros(np.sum(free))
            r = -g_free.copy()
            p = r.copy()
            rr = np.dot(r, r)
            for _ in range(maxcg):
                Hp = Hv_free(p)
                pHp = np.dot(p, Hp)
                if pHp <= 1e-14:
                    if np.linalg.norm(d_free) < 1e-14:
                        d_free = -g_free.copy()
                    break
                alpha_cg = rr / pHp
                d_free = d_free + alpha_cg * p
                r_new = r - alpha_cg * Hp
                rr_new = np.dot(r_new, r_new)
                if np.sqrt(rr_new) <= tol_cg:
                    r = r_new
                    rr = rr_new
                    break
                beta = rr_new / rr
                p = r_new + beta * p
                r = r_new
                rr = rr_new
            d[free] = d_free
        step = alpha
        F_w = F_batch(w, indices_S)
        sg_d = np.dot(sg, d)
        if sg_d >= 0:
            d = -sg
            sg_d = -np.dot(sg, sg)
        w_new = w.copy()
        for _ in range(20):
            w_trial = project_orthant(w + step * d, z)
            if F_batch(w_trial, indices_S) <= F_w + sigma * step * sg_d:
                w_new = w_trial
                break
            step *= 0.5
            if step < 1e-12:
                w_new = w.copy()
                break
        w = w_new
        indices_new = np.random.choice(N, size=n, replace=False)
        grads_new = np.array([grad_i(w, i) for i in indices_new])
        g_new = np.mean(grads_new, axis=0)
        if n > 1:
            var_vec = np.var(grads_new, axis=0, ddof=1)
        else:
            var_vec = np.zeros_like(g_new)
        V_norm1 = np.sum(var_vec)
        gg_new = np.dot(g_new, g_new)
        if gg_new > 1e-16:
            if V_norm1 / n > theta**2 * gg_new:
                n_new = int(np.ceil(V_norm1 / (theta**2 * gg_new))) + 1
                n = min(n_new, N)
        history.append(w.copy().tolist())
        batch_sizes.append(n)
        if np.linalg.norm(grad_full(w)) < 1e-6:
            break
    return history, batch_sizes

#### Listato B.4 — BB-CCV (Barzilai–Borwein con campionamento dinamico)

In [ ]:
def bb_dynamic_gd(w0, theta, max_iter, alpha, batch0):
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history     = [w.copy().tolist()]
    batch_sizes = [n]

    w_prev = w.copy()
    g_prev = None
    for k in range(max_iter):
        indices = np.random.choice(N, size=n, replace=False)
        grads = np.array([grad_i(w, i) for i in indices])
        g = np.mean(grads, axis=0)
        if k > 0 and g_prev is not None:
            s = w - w_prev
            y = g - g_prev
            sy = np.dot(s, y)
            if abs(sy) > 1e-14:
                step_bb = np.dot(s, s) / sy
                step = np.clip(step_bb, alpha / 20.0, alpha * 5.0)
            else:
                step = alpha
        else:
            step = alpha

        w_prev = w.copy()
        g_prev = g.copy()
        def J_batch(w_curr):
            return np.mean([loss_i(w_curr, i) for i in indices])

        c1 = 1e-4
        J_curr = J_batch(w)
        g_norm2 = np.dot(g, g)

        if g_norm2 > 1e-16:
            for _ in range(30):
                w_new = w - step * g
                if J_batch(w_new) <= J_curr - c1 * step * g_norm2:
                    break
                step *= 0.5
            else:
                step = 0.0

        w = w - step * g
        if n > 1:
            var_vec = np.var(grads, axis=0, ddof=1)
        else:
            var_vec = np.zeros_like(g)
        V_norm1 = np.sum(var_vec)

        gg = np.dot(g, g)
        if gg > 1e-16:
            if V_norm1 / n > theta**2 * gg:
                n_new = int(np.ceil(V_norm1 / (theta**2 * gg))) + 1
                n = min(n_new, N)
        history.append(w.copy().tolist())
        batch_sizes.append(n)

        if np.linalg.norm(grad_full(w)) < 1e-6:
            break

    return history, batch_sizes

In [ ]:
#@title 4. Esecuzione dei quattro metodi (listati B.1-B.4) + riferimento sklearn
# Tempo stimato (Colab): ~15-25 min con MAX_ITER=300 (riproduzione esatta).
# Per una prova rapida imposta MAX_ITER = 60.
SEED, THETA, ALPHA, BATCH0, MAX_ITER = 42, 0.5, 1.0, 64, 300
NU, SIGMA, MAXCG = 1e-3, 1e-4, 50
w0 = np.zeros(C * Xtr_aug.shape[1])
Xte_aug = np.hstack([Xte, np.ones((Xte.shape[0], 1))])

def _W(w):
    return w.reshape(C, -1)

def acc_test(w):
    return float(np.mean(np.argmax(Xte_aug @ _W(w).T, axis=1) == Yte))

def loss_full_l1(w):
    return float(np.mean([loss_i(w, i) for i in range(N)])) + NU * np.sum(np.abs(w))

def subgrad_full_l1(w):
    gJ = grad_full(w)                      # LAM = 0 per il metodo L1
    g = np.zeros_like(w)
    for i in range(len(w)):
        if w[i] > 0:        g[i] = gJ[i] + NU
        elif w[i] < 0:      g[i] = gJ[i] - NU
        elif gJ[i] < -NU:   g[i] = gJ[i] + NU
        elif gJ[i] > NU:    g[i] = gJ[i] - NU
        else:               g[i] = 0.0
    return g

methods = [
    ("Dynamic GD",   lambda: dynamic_gd(w0, THETA, MAX_ITER, ALPHA, BATCH0)),
    ("Newton-CG",    lambda: newton_cg(w0, THETA, MAX_ITER, ALPHA, BATCH0, R, MAXCG)),
    ("Newton-CG L1", lambda: newton_l1(w0, THETA, MAX_ITER, ALPHA, BATCH0, NU, SIGMA, MAXCG)),
    ("BB-CCV",       lambda: bb_dynamic_gd(w0, THETA, MAX_ITER, ALPHA, BATCH0)),
]

results, accs, batches, pesi = {}, {}, {}, {}
for name, run in methods:
    np.random.seed(SEED)
    LAM = 0.0 if name == "Newton-CG L1" else 1e-4    # regolarizzazione
    t0 = time.time()
    history, batch_sizes = run()
    dt = time.time() - t0
    wf = np.array(history[-1])
    pesi[name] = wf.copy()                            # per la cella predizioni
    if name == "Newton-CG L1":
        loss_f = loss_full_l1(wf)
        gnorm = float(np.linalg.norm(subgrad_full_l1(wf)))
        nnz = int(np.sum(wf != 0))
    else:
        loss_f = float(np.mean([loss_i(wf, i) for i in range(N)]))
        gnorm = float(np.linalg.norm(grad_full(wf)))
        nnz = None
    results[name] = dict(iter=len(history)-1, acc=acc_test(wf), loss=loss_f,
                         gnorm=gnorm, batch=max(batch_sizes), batch0=batch_sizes[0],
                         time=dt, nnz=nnz)
    accs[name]  = [acc_test(np.array(w)) for w in history]
    batches[name] = batch_sizes
    print(f"{name:14s} iter={results[name]['iter']:3d}  "
          f"acc={results[name]['acc']*100:5.2f}%  ||grad||={gnorm:.1e}  "
          f"tempo={dt:5.1f}s" + (f"  nnz={nnz}" if nnz is not None else ""))

# --- riferimento scikit-learn (L-BFGS) ---
from sklearn.linear_model import LogisticRegression
m = LogisticRegression(max_iter=2000, C=1.0)
m.fit(Xtr, Ytr)
acc_ref = m.score(Xte, Yte)
print(f"Riferimento sklearn (L-BFGS): acc = {acc_ref*100:.2f}%")
results["sklearn (L-BFGS)"] = dict(acc=acc_ref)

In [ ]:
#@title 4b. Salva i pesi addestrati e le features di test (per nsynth_nota_test)
# Salva i pesi finali dei 4 metodi (variabile `pesi` della cella precedente)
# insieme alla standardizzazione (mu, sd) in pesi_nota.npz, e le features
# di test gia' standardizzate (Xte, Yte, nomi delle clip) in features_nota.npz.
# Servono al notebook nsynth_nota_test (test su clip audio qualsiasi o su
# esempi del test set NSynth).
import json as _json

names_test = list(_json.load(open("nsynth-test/examples.json")).keys())

np.savez_compressed(
    "pesi_nota.npz",
    **{"Dynamic_GD":  pesi["Dynamic GD"],
       "Newton-CG":   pesi["Newton-CG"],
       "Newton-CG_L1": pesi["Newton-CG L1"],
       "BB-CCV":      pesi["BB-CCV"],
       "mu": mu, "sd": sd, "C": C, "NOTE": np.array(NOTE)})
np.savez_compressed(
    "features_nota.npz",
    Xte=Xte, Yte=Yte, names=np.array(names_test))

print("Salvati:")
for f in ("pesi_nota.npz", "features_nota.npz"):
    print(" -", f, "con chiavi:",
          {k: v.shape for k, v in np.load(f).items()})
try:
    from google.colab import files
    files.download("pesi_nota.npz")
    files.download("features_nota.npz")
except ImportError:
    print("Non sei in Colab: i file sono nella cartella corrente.")


In [ ]:
#@title 5. Figura 1 — accuratezza sul test vs iterazioni (come nella tesi)
COLORS = {"Dynamic GD": "#1f77b4", "Newton-CG": "#d62728",
          "Newton-CG L1": "#2ca02c", "BB-CCV": "#9467bd"}
ks = np.arange(0, MAX_ITER + 1)

fig, ax = plt.subplots(figsize=(6.4, 4.2))
for name, _ in methods:
    a = np.asarray(accs[name]) * 100
    ax.plot(ks[:len(a)], a, lw=1.8, label=name, color=COLORS[name])
ax.axhline(100.0 / 12, color="gray", ls="--", lw=1.2, label="Casuale (1/12)")
ax.set_xlabel(r"Iterazione $k$")
ax.set_ylabel(r"Accuratezza sul test (\%)")
ax.set_title("NSynth nota: accuratezza di test vs iterazioni")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("nota_accuracy.png", dpi=150)
fig.savefig("nota_accuracy.pdf")
plt.show()
print("Salvate: nota_accuracy.png / .pdf")

In [ ]:
#@title 6. Figura 2 — dinamica del batch n_k vs k (come nella tesi)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
for name, _ in methods:
    b = np.asarray(batches[name])
    ax.step(np.arange(len(b)), b, where="mid", lw=1.6, label=name,
            color=COLORS[name])
ax.set_xlabel(r"Iterazione $k$")
ax.set_ylabel(r"Dimensione del batch $n_k$")
ax.set_title("NSynth nota: dinamica del batch (CCV)")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("nota_batch.png", dpi=150)
fig.savefig("nota_batch.pdf")
plt.show()
print("Salvate: nota_batch.png / .pdf")

In [ ]:
#@title 7. Tabella dei risultati (pronta per il LaTeX) + riferimento
print("=== Tabella 6.5 nota (risultati) ===")
print(r"Metodo & Acc. test & $\|\nabla J(w)\|_2$ & Batch finale & Tempo (s) & Coeff. non nulli \\")
for name, _ in methods:
    r = results[name]
    nnz = f"{r['nnz']}/300" if r["nnz"] is not None else "---"
    print(f"{name} & {r['acc']*100:.1f}\\% & ${r['gnorm']:.1e}$ & "
          f"{r['batch']:,} & {r['time']:.1f} & {nnz} \\\\")

print(f"Riferimento sklearn (L-BFGS) & {results['sklearn (L-BFGS)']['acc']*100:.1f}\\% & --- & --- & --- & --- \\\\")

In [ ]:
#@title 8. (Opzionale) Scarica le figure e i risultati
# In Colab: clicca i file generati o esegui questa cella per scaricarli.
try:
    from google.colab import files
    import json as _json
    _json.dump(results, open("results.json", "w"), indent=2)
    for f in ("nota_accuracy.png", "nota_accuracy.pdf",
              "nota_batch.png", "nota_batch.pdf", "results.json"):
        files.download(f)
except ImportError:
    print("Non sei in Colab: i file sono nella cartella corrente.")
    print(os.listdir("."))

In [ ]:
#@title 8. (Facoltativo) Predizioni di un esempio con audio (nomi delle note)
# Ascolti la nota e vedi cosa avrebbe dovuto dire il modello e cosa dice
# ciascun metodo (i nomi delle note: C, C#, D, ...).
from IPython.display import Audio
import json, random

names_test = list(json.load(open("nsynth-test/examples.json")).keys())

# --- scegli l'esempio: cambia qui ---
nota = None                      # es. "C", "F#", "A", ...  oppure None (casuale)
if nota:
    i = random.choice([k for k, n in enumerate(names_test)
                       if NOTE[json.load(open("nsynth-test/examples.json"))[n]["pitch"] % 12] == nota])
else:
    i = random.randrange(len(names_test))


pitch_reale = json.load(open("nsynth-test/examples.json"))[names_test[i]]["pitch"]
print(f"Esempio {i}: {names_test[i]}  |  Nota reale: {NOTE[pitch_reale % 12]} (MIDI {pitch_reale})")
for name, w in pesi.items():
    z = _W(w) @ Xte_aug[i]
    e = np.exp(z - np.max(z)); p = e / e.sum()
    top = np.argsort(p)[::-1]
    ent = -np.sum(p * np.log(p + 1e-15)) / np.log(len(p))  # 0 = certo, 1 = uniforme
    pred = NOTE[top[0]]
    segno = '✔' if pred == NOTE[pitch_reale % 12] else '✘'
    print(f"  {name:14s}: {pred} ({p[top[0]]*100:4.1f}%) {segno} | "
          f"2°: {NOTE[top[1]]} ({p[top[1]]*100:4.1f}%) | "
          f"3°: {NOTE[top[2]]} ({p[top[2]]*100:4.1f}%) | incertezza {ent*100:3.0f}%")
display(Audio(f"nsynth-test/audio/{names_test[i]}.wav"))